<style>
  .cell-markdown { overflow: auto !important; }
  .mermaid { max-width: 100%; height: auto; }
</style>

# Hopping Windows Walkthrough


## Overview

[Preparation](#prep)

* [Topology](#topology)
* [Steps](#steps)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
import sys
sys.path.insert(1, "../..")
sys.path.insert(1, "../../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"

#

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)


---
<a id="topology"></a>
## Topology

Now it's time for the walkthrough itself. We go for a slightly simpler example for the walkthroughs.

The corresponding test can be found here: [test_windows.py](../../../../test/streams/test_windows.py)

In [3]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = 100
hop_int = size_int // 2
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    # 1. Select customer_id, price and ts from the value.
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    # 2. Expire with window size order_generator.ts_step_int * 100, hop size 100 / 2 = 50 and allowed_lateness = window_size * 2,  
    .expire_hopping(lambda r: r["ts"], size_int, hop_int, allowed_lateness_int)
    # 3. Deduplicate.
    .distinct()
)
#
# 4. Set up the window: group by customer ID, count the orders, sum up the prices of the orders and get the last timestamp of the window.
sink_tn = order_tn.group_by_agg_hopping(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    hop_int=hop_int,
    key_fun=lambda r: r["customer_id"],
    agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                              "total_price": agg_r["total_price"] + r["price"],
                              "last_ts": max(agg_r["last_ts"], r["ts"])},
    agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0},
    project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                        "orders": agg_r["orders"],
                                        "total_price": agg_r["total_price"],
                                        "last_ts": agg_r["last_ts"]},
    trigger_positive_only_bool=False
).sink(sink_str)                                       
#
_ = tn = Tn.build(sink_tn)


What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_hopping()`: Expire with window size `100`, hop size `100 // 2 = 50` and `allowed_lateness` = `size_int * 2 = 200`.  
3. `distinct()`: Deduplicate.
4. `group_by_agg_hoppingg()`: Set up the hopping window: group by customer ID, count the orders sum up the prices of the orders and get the last timestamp of the window.

Next, we illustrate how the hopping window works by processing some example data - one by one, in baby steps.

We use two types of illustrations in each step:
1. Top-down view:
  * time proceeds from top to bottom (starting with `0`)
  * small grey circles mark the time every `100` ms for clarity
  * the latest timestamp of the input after the respective step is written at the top 
  * new events coming in a step are indicated a blue frame
  * old events have a grey frame
  * the triggered outputs in the sink are indicated by green color
2. Left-right view:
  * time proceeds from left to right (starting with `0`)
  * new windows appear below the old windows
  * `^`: latest timestamp of this step
  * `(^)`: latest timestamp of the previous step
  * `<<<`: time window(s) containing the event from this step
  * `!!!`: time window(s) triggered by the event from this step


<a id="steps"></a>
## Steps

### Step 1

In step 1, first, the first order arrives from customer 1 at timestamp 10:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 10"]
        direction TB
        0(("0")) e1@-.-> 10
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `0` (start, visualized by `(^)`) to `10` (visualized by `^`),
* and falls into the first hopping window `[0, 100)`, i.e., from `0` until `99` (visualized by `<<<`):
```
[0 ---------- 100) <<<
(^) 
   ^
```

As the latest timestamp is not yet beyond the end of the first hopping window, no output is triggered.

Let's see this happening for real:

In [4]:
process(tn, customer_id=1, price=100, ts=10, w=1)


Triggers:


### Step 2

In baby step 2, the second order arrives from customer 1 at timestamp 50:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 50"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `10` to `50`,
* and falls into the tumbling windows `[0, 100)` and `[50, 150)`:
```
[0 ---------- 100) <<<
       [50 ---------- 150) <<<
  (^)   
        ^
```

The latest timestamp is however still not beyond the end of the first hopping window `[0, 100)`, so again, no output is triggered:

In [5]:
process(tn, customer_id=1, price=200, ts=50, w=1)


Triggers:


### Step 3

In step 3, an order from customer arrives shortly after the end of the first tumbling window:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 300,\n&quot;last_ts&quot;: 50\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    105 --- space1
    linkStyle 4 stroke:none
    105 Link@== Triggers ==> Output
    linkStyle 5 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp to from `50` to `105`,
* falls into the two windows `[50, 150)` and `[100, 200)`,
* and triggers the window `[0, 100)` (visualized by `!!!`):
```
[0 ---------- 100) !!!
       [50 ---------- 150) <<<
             [100 ---------- 200) <<<
        (^)       
                  ^
```

The triggered window `[0, 100)` contains the aggregation of the two orders from customer 1 that came in between `[0, 100)`, because `latest_ts = 105 >= 100` evaluates to `True` in `trigger_fun`.

In [6]:
process(tn, customer_id=2, price=50, ts=105, w=1)

Triggers:
{'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50, 'window_end': 100}


### Step 4

Step 4 shows the effect of a retraction coming in (weight = `-1`) as the order from customer 1 at timestamp `50` is canceled:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s style='color:red;'>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#bb0000,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100,\n&quot;last_ts&quot;: 10\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    50 Link@== Triggers ==> Output
    linkStyle 4 stroke:#00bb00,stroke-width:3px;
```

This event:
* does not advance latest timestamp since it has weight `-1` (the latest timestamp stays at `105`),
* falls into window `[0, 100)`,
* and triggers the correction of the passed window `[0, 100)`:
```
[0 ---------- 100) <<< !!!
       [50 ---------- 150)
             [100 ---------- 200)
                 (^)
                  ^       
```

The correction still comes in early enough as in still within the `allowed_lateness = 200` (actually, it comes in during the *window buffer time* already): `105 (latest) - ( 150 (end of the last/second hopping window for ts = 50) + 100 (window buffer) = 250 ) = -145 < 200`


In [7]:
process(tn, customer_id=1, price=200, ts=50, w=-1)


Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10, 'window_end': 100}


### Step 5

An order from customer 3 at timestamp `150`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 150"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 150
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 99,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 150}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    150 --- space1
    linkStyle 5 stroke:none
    150 Link@== Triggers ==> Output
    linkStyle 6 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `105` to `150`,
* falls into the two windows `[100, 200)` and `[150, 250)`,
* and triggers the passed window `[50, 150)`:
```
[0 ---------- 100)
       [50 ---------- 150) !!!
             [100 ---------- 200) <<<
                      [150 ---------- 250) <<<
                 (^)
                        ^
```


In [8]:
process(tn, customer_id=3, price=99, ts=150, w=1)

Triggers:
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 105, 'window_end': 150}


### Step 6

An order from customer 3 at timestamp `250`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 250"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 150 e6@-.-> 250
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 99,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    250@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 67,\n&quot;ts&quot;: 250}"}
    style 250 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50,\n&quot;last_ts&quot;: 105\n&quot;window_end&quot;: 200}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    250 Link@== Triggers ==> Output1
    linkStyle 6 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 99,\n&quot;last_ts&quot;: 150\n&quot;window_end&quot;: 200}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    250 Link@== Triggers ==> Output2
    linkStyle 7 stroke:#00bb00,stroke-width:3px

    Output3@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 99,\n&quot;last_ts&quot;: 150\n&quot;window_end&quot;: 250}"}
    style Output3 fill:none,stroke:#00bb00,stroke-width:6px
    250 Link@== Triggers ==> Output3
    linkStyle 8 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp to `250`,
* falls into the windows `[200, 300)` and `[250, 300)`,
* and triggers the passed windows `[100, 200)` and `[150, 250)`:
```
[0 ---------- 100)
       [50 ---------- 150)
             [100 ---------- 200) !!!
                     [150 ---------- 250) !!!
                            [200 ---------- 300) <<<
                                    [250 ---------- 350) <<<
                      (^)             
                                      ^
```


In [9]:
process(tn, customer_id=2, price=67, ts=250, w=1)


Triggers:
{'customer_id': 3, 'orders': 1, 'total_price': 99, 'last_ts': 150, 'window_end': 200}
{'customer_id': 3, 'orders': 1, 'total_price': 99, 'last_ts': 150, 'window_end': 250}
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 105, 'window_end': 200}


### Step 7

An order from customer 3 arrives too late (but not too late) at timestamp `170`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 250"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 150 e6@-.-> 170 e7@-.-> 250
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 99,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    250@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 67,\n&quot;ts&quot;: 250}"}
    style 250 fill:none,stroke:#333,stroke-width:6px

    170@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 170}"}
    style 170 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 199,\n&quot;last_ts&quot;: 170\n&quot;window_end&quot;: 200}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    170 Link@== Triggers ==> Output1
    linkStyle 7 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 199,\n&quot;last_ts&quot;: 170\n&quot;window_end&quot;: 250}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    170 Link@== Triggers ==> Output2
    linkStyle 8 stroke:#00bb00,stroke-width:3px
```

This event:
* does not advance the latest timestamp (it stays at `250`),
* falls into the windows `[100, 200)` and `[150, 200)`,
* and triggers corrections for the passed windows `[100, 200)` and `[150, 250)` (because it falls into them):
```
[0 ---------- 100)
       [50 ---------- 150)
             [100 ---------- 200) <<< !!!
                     [150 ---------- 250) <<< !!!
                            [200 ---------- 300)
                                    [250 ---------- 350)
                                     (^)
                                      ^
```


In [10]:
process(tn, customer_id=3, price=100, ts=170, w=1)

Triggers:
{'customer_id': 3, 'orders': 2, 'total_price': 199, 'last_ts': 170, 'window_end': 200}
{'customer_id': 3, 'orders': 2, 'total_price': 199, 'last_ts': 170, 'window_end': 250}


### Step 8

An order from customer 1 arrives at timestamp `450`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 450"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105 e5@-.-> 150 e6@-.-> 170 e7@-.-> 250 e8@-.-> 450
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 99,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    250@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 67,\n&quot;ts&quot;: 250}"}
    style 250 fill:none,stroke:#333,stroke-width:6px

    170@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 170}"}
    style 170 fill:none,stroke:#333,stroke-width:6px

    450@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 10,\n&quot;ts&quot;: 450}"}
    style 450 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 67,\n&quot;last_ts&quot;: 250\n&quot;window_end&quot;: 300}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    450 Link@== Triggers ==> Output1
    linkStyle 8 stroke:#00bb00,stroke-width:3px

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 67,\n&quot;last_ts&quot;: 250\n&quot;window_end&quot;: 350}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    450 Link@== Triggers ==> Output2
    linkStyle 9 stroke:#00bb00,stroke-width:3px
```
This event:
* advances the latest timestamp to `450`,
* falls into the windows `[400, 500)` and `[450, 550)`,
* and triggers corrections for the passed windows `[200, 300)` and `[250, 350)`:
```
[0 ---------- 100)
       [50 ---------- 150)
             [100 ---------- 200)
                     [150 ---------- 250)
                            [200 ---------- 300) !!!
                                    [250 ---------- 350) !!!
                                           [300 ---------- 400)
                                                   [350 ---------- 450)
                                                          [400 ---------- 500) <<<
                                                                  [450 ---------- 550) <<<
                                     (^)
                                                                    ^
```


This event triggers corrections of the tumbling windows `[200, 300)` and `[250, 350)`.


In [11]:
process(tn, customer_id=1, price=10, ts=450, w=1)

Triggers:
{'customer_id': 2, 'orders': 1, 'total_price': 67, 'last_ts': 250, 'window_end': 300}
{'customer_id': 2, 'orders': 1, 'total_price': 67, 'last_ts': 250, 'window_end': 350}


### Step 9

An order from customer 1 arrives too late at timestamp `20`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 450"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 20 e3@-.-> 50 e4@-.-> 100(("100")) e5@-.-> 105 e6@-.-> 150 e7@-.-> 170 e8@-.-> 250 e9@-.-> 450
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 99,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    250@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 67,\n&quot;ts&quot;: 250}"}
    style 250 fill:none,stroke:#333,stroke-width:6px

    170@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 170}"}
    style 170 fill:none,stroke:#333,stroke-width:6px

    450@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 10,\n&quot;ts&quot;: 450}"}
    style 450 fill:none,stroke:#1565c0,stroke-width:6px

    20@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 999,\n&quot;ts&quot;: 20}"}
    style 450 fill:none,stroke:#1565c0,stroke-width:6px
```
This event:
* does not advances the latest timestamp (it stays at `450`),
* falls into the window `[0, 100)`,
* but comes in too late: (`allowed_lateness = 200`): `450 (latest) - ( 100 (window end for the last/second hopping window for ts = 20) + 100 (window buffer) = 200 ) = 250 > 200`, i.e., it is discarded right away:
```
[0 ---------- 100) <<<
       [50 ---------- 150)
             [100 ---------- 200)
                     [150 ---------- 250)
                            [200 ---------- 300)
                                    [250 ---------- 350)
                                           [300 ---------- 400)
                                                   [350 ---------- 450)
                                                          [400 ---------- 500) <<<
                                                                  [450 ---------- 550) <<<
                                                                   (^)
                                                                    ^
```


In [12]:
process(tn, customer_id=1, price=10, ts=20, w=1)

Triggers:
